# SparseGPT GSM8K Results Analysis

比較 SparseGPT 稀疏化模型與原始模型、純量化模型在 GSM8K 上的表現。

分析包含：
1. **Baseline vs Sparse** — 稀疏化對準確率的影響
2. **Quantized-only vs Sparse+Quantized** — 稀疏化 + 量化 vs 純量化
3. **Sparse 之間比較** — 不同稀疏率 (20% vs 30%) + 不同量化方法的比較

In [37]:
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

def style_change_gradient(styler, df):
    """對所有含 'Change' 的欄位上背景漸層色。
    - Acc Change: 綠色=好(正), 紅色=差(負) → RdYlGn
    - GPU Change: 綠色=好(負), 紅色=差(正) → RdYlGn_r
    """
    change_cols = [c for c in df.columns if 'Change' in c and c in styler.columns]
    for col in change_cols:
        vals = pd.to_numeric(df[col], errors='coerce').dropna().values
        if len(vals) == 0:
            continue
        abs_max = max(abs(np.nanmin(vals)), abs(np.nanmax(vals)), 0.01)

        if 'Acc' in col:
            cmap = 'RdYlGn'      # positive=green, negative=red
        else:
            cmap = 'RdYlGn_r'    # negative=green (GPU reduction is good)

        styler = styler.background_gradient(
            cmap=cmap, subset=[col],
            vmin=-abs_max, vmax=abs_max
        )
    return styler

In [38]:
import re

results_dir = Path('../results') if Path('../results').exists() else Path('results')

def scan_results(results_dir):
    """自動掃描 results 目錄，從目錄名和 JSON 解析模型配置。"""
    rows = []
    base_re = re.compile(r'^(Llama-[\d.]+-(\d+B)-Instruct)')
    sparse_re = re.compile(r'-llmc-sparsegpt-(sparse(\d+)|2x4)-\d{8}_\d{6}')
    quant_re = re.compile(r'-(awq|gptq|bnb)-(\d+)bit-\d{8}_\d{6}')

    for d in sorted(results_dir.iterdir()):
        if not d.is_dir():
            continue
        f = d / 'gsm8k_results.json'
        if not f.exists():
            continue

        name = d.name
        bm = base_re.match(name)
        if not bm:
            continue

        model_size = bm.group(2)
        rest = name[bm.end():]

        # --- Sparsity ---
        sparsity = None
        sm = sparse_re.search(rest)
        if sm:
            sparsity = '2:4' if sm.group(1) == '2x4' else f'{sm.group(2)}%'
            rest_q = rest[sm.end():]
        else:
            rest_q = rest

        # --- Quantization ---
        qm = quant_re.search(rest_q)
        quant_method = qm.group(1).upper() if qm else None
        quant_bits = qm.group(2) if qm else None

        # --- Load JSON ---
        with open(f) as fh:
            data = json.load(fh)
        qcfg = data.get('quantization_config') or {}
        gs = qcfg.get('group_size')

        # --- Build columns ---
        quant_col = 'BNB (NF4)' if quant_method == 'BNB' else (quant_method or 'None')
        q_short = quant_method or ''

        if sparsity and quant_method:
            method = 'Sparse+Quant'
            gs_tag = f' (gs{gs})' if gs else ''
            label = f'Sparse {sparsity} + {q_short}{gs_tag}'
        elif sparsity:
            method = 'Sparse-only'
            label = f'Sparse {sparsity}'
        elif quant_method:
            method = 'Quantize-only'
            label = f'{q_short} {quant_bits}-bit'
        else:
            method = 'Baseline'
            label = 'Baseline (FP16)'

        rows.append({
            'Label': label, 'Model': model_size, 'Method': method,
            'Sparsity': sparsity or 'None',
            'Quantization': quant_col,
            'Group Size': gs if gs else '-',
            'Accuracy (%)': round(data['accuracy'] * 100, 2),
            'Correct': data['correct'], 'Total': data['total'],
            'GPU Peak (MB)': round(data.get('gpu_peak_mb', float('nan')), 2),
            'Throughput (tok/s)': round(data.get('throughput_tokens_per_sec', float('nan')), 2),
            'Time (s)': round(data.get('total_generation_time_sec', float('nan')), 2),
            'Energy (kWh)': data.get('energy_consumed_kwh', float('nan')),
            'CO2 (kg)': data.get('emissions_kg_co2', float('nan')),
        })

    return pd.DataFrame(rows)

print('Functions defined.')

Functions defined.


In [39]:
# 自動掃描 results 目錄
df_scan = scan_results(results_dir)

# 按模型大小分組（依參數量排序）
model_sizes = sorted(df_scan['Model'].unique(), key=lambda x: int(x.replace('B', '')))
models = {}
for ms in model_sizes:
    models[ms] = df_scan[df_scan['Model'] == ms].reset_index(drop=True)

for ms, df in models.items():
    print(f'  Llama-{ms}-Instruct: {len(df)} configurations')
print(f'\nTotal: {len(df_scan)} configurations across {len(model_sizes)} model sizes')

  Llama-1B-Instruct: 14 configurations
  Llama-3B-Instruct: 9 configurations
  Llama-8B-Instruct: 4 configurations

Total: 27 configurations across 3 model sizes


---
## 1. 各模型完整比較表

In [40]:
raw_cols = ['Label', 'Sparsity', 'Quantization', 'Group Size',
            'Accuracy (%)', 'GPU Peak (MB)', 'Throughput (tok/s)', 'Time (s)',
            'Energy (kWh)', 'CO2 (kg)']

raw_fmt = {'Accuracy (%)': '{:.2f}', 'GPU Peak (MB)': '{:.1f}',
           'Throughput (tok/s)': '{:.1f}', 'Time (s)': '{:.1f}',
           'Energy (kWh)': '{:.5f}', 'CO2 (kg)': '{:.5f}'}

for ms, df in models.items():
    baseline_acc = df.loc[df['Method'] == 'Baseline', 'Accuracy (%)'].values[0]
    print(f'=== Llama-{ms}-Instruct GSM8K (Baseline Acc: {baseline_acc}%) ===')
    display(df[raw_cols].style
        .format(raw_fmt, na_rep='-')
        .set_caption(f'Llama-{ms}-Instruct - GSM8K Results')
    )
    print()

=== Llama-1B-Instruct GSM8K (Baseline Acc: 34.87%) ===


,Label,Sparsity,Quantization,Group Size,Accuracy (%),GPU Peak (MB),Throughput (tok/s),Time (s),Energy (kWh),CO2 (kg)
0,Baseline (FP16),None,None,-,34.87,2482.4,2738.4,546.3,0.03835,0.02463
1,AWQ 4-bit,None,AWQ,128,25.85,1128.1,1634.3,910.4,-,-
2,AWQ 4-bit,None,AWQ,64,27.37,1146.2,1630.2,915.4,0.04551,0.02923
3,BNB 4-bit,None,BNB (NF4),-,31.69,1145.1,1314.8,1141.8,-,-
4,GPTQ 4-bit,None,GPTQ,128,30.02,1128.6,1506.5,988.2,-,-
5,Sparse 20%,20%,None,-,30.86,2478.7,2394.2,624.0,0.04299,0.02761
6,Sparse 20% + AWQ (gs64),20%,AWQ,64,29.64,1146.2,1542.8,971.4,0.04785,0.03074
7,Sparse 20% + GPTQ (gs32),20%,GPTQ,32,28.20,1182.9,775.4,1925.4,-,-
8,Sparse 30%,30%,None,-,26.69,2478.7,3031.4,490.1,0.03339,0.00013
9,Sparse 30% + AWQ (gs64),30%,AWQ,64,23.05,1146.2,639.9,2319.4,0.17478,0.11227



=== Llama-3B-Instruct GSM8K (Baseline Acc: 68.01%) ===


,Label,Sparsity,Quantization,Group Size,Accuracy (%),GPU Peak (MB),Throughput (tok/s),Time (s),Energy (kWh),CO2 (kg)
0,Baseline (FP16),None,None,-,68.01,6348.5,515.2,2965.1,0.21901,0.14069
1,AWQ 4-bit,None,AWQ,128,67.40,2414.1,731.9,2085.6,-,-
2,BNB 4-bit,None,BNB (NF4),-,65.66,2393.9,702.1,2176.1,-,-
3,GPTQ 4-bit,None,GPTQ,128,64.97,2417.0,654.0,2336.5,-,-
4,Sparse 20%,20%,None,-,65.96,6348.5,333.3,4570.8,0.34481,0.22150
5,Sparse 30%,30%,None,-,61.26,6348.5,336.4,4505.7,0.33969,0.21821
6,Sparse 30% + AWQ (gs64),30%,AWQ,64,59.06,2466.6,599.4,2524.7,-,-
7,Sparse 30% + GPTQ (gs64),30%,GPTQ,64,60.20,2469.5,692.3,2186.5,-,-
8,Sparse 40%,40%,None,-,54.89,6348.5,346.0,4364.4,0.32892,0.21129



=== Llama-8B-Instruct GSM8K (Baseline Acc: 76.57%) ===


,Label,Sparsity,Quantization,Group Size,Accuracy (%),GPU Peak (MB),Throughput (tok/s),Time (s),Energy (kWh),CO2 (kg)
0,Baseline (FP16),None,None,-,76.57,15605.5,503.5,3051.0,-,-
1,AWQ 4-bit,None,AWQ,128,73.92,5831.3,589.4,2599.2,-,-
2,BNB 4-bit,None,BNB (NF4),-,75.82,5807.4,569.7,2691.1,-,-
3,GPTQ 4-bit,None,GPTQ,128,75.51,5836.2,292.7,5228.4,-,-


---
## 2. 變化量分析 (相對 Baseline)

計算每種方法相對於 FP16 Baseline 的 Accuracy / GPU / Latency 變化百分比。

In [41]:
def add_change_cols(df):
    """Add accuracy / GPU / latency change (%) columns relative to baseline."""
    df = df.copy()
    bl_acc = df.loc[df['Method'] == 'Baseline', 'Accuracy (%)'].values[0]
    bl_gpu = df.loc[df['Method'] == 'Baseline', 'GPU Peak (MB)'].values[0]
    bl_time = df.loc[df['Method'] == 'Baseline', 'Time (s)'].values[0]
    df['Acc Change (%)'] = ((df['Accuracy (%)'] - bl_acc) / bl_acc * 100).round(2)
    df['GPU Change (%)'] = ((df['GPU Peak (MB)'] - bl_gpu) / bl_gpu * 100).round(2)
    df['Latency Change (%)'] = ((df['Time (s)'] - bl_time) / bl_time * 100).round(2)
    return df

models_chg = {ms: add_change_cols(df) for ms, df in models.items()}

chg_cols = ['Label', 'Accuracy (%)', 'Acc Change (%)',
            'GPU Peak (MB)', 'GPU Change (%)',
            'Time (s)', 'Latency Change (%)']
chg_fmt = {
    'Accuracy (%)': '{:.2f}', 'Acc Change (%)': '{:+.2f}',
    'GPU Peak (MB)': '{:.1f}', 'GPU Change (%)': '{:+.2f}',
    'Time (s)': '{:.1f}', 'Latency Change (%)': '{:+.2f}'
}

for ms, df in models_chg.items():
    print(f'=== {ms} Model - Accuracy, GPU & Latency Change vs Baseline ===')
    s = df[chg_cols].style.format(chg_fmt)
    display(style_change_gradient(s, df[chg_cols]))
    print()

=== 1B Model - Accuracy, GPU & Latency Change vs Baseline ===


,Label,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%)
0,Baseline (FP16),34.87,+0.00,2482.4,+0.00,546.3,+0.00
1,AWQ 4-bit,25.85,-25.87,1128.1,-54.55,910.4,+66.66
2,AWQ 4-bit,27.37,-21.51,1146.2,-53.82,915.4,+67.57
3,BNB 4-bit,31.69,-9.12,1145.1,-53.87,1141.8,+109.02
4,GPTQ 4-bit,30.02,-13.91,1128.6,-54.54,988.2,+80.90
5,Sparse 20%,30.86,-11.50,2478.7,-0.15,624.0,+14.23
6,Sparse 20% + AWQ (gs64),29.64,-15.00,1146.2,-53.82,971.4,+77.82
7,Sparse 20% + GPTQ (gs32),28.20,-19.13,1182.9,-52.35,1925.4,+252.46
8,Sparse 30%,26.69,-23.46,2478.7,-0.15,490.1,-10.28
9,Sparse 30% + AWQ (gs64),23.05,-33.90,1146.2,-53.82,2319.4,+324.60



=== 3B Model - Accuracy, GPU & Latency Change vs Baseline ===


,Label,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%)
0,Baseline (FP16),68.01,+0.00,6348.5,+0.00,2965.1,+0.00
1,AWQ 4-bit,67.40,-0.90,2414.1,-61.97,2085.6,-29.66
2,BNB 4-bit,65.66,-3.46,2393.9,-62.29,2176.1,-26.61
3,GPTQ 4-bit,64.97,-4.47,2417.0,-61.93,2336.5,-21.20
4,Sparse 20%,65.96,-3.01,6348.5,+0.00,4570.8,+54.15
5,Sparse 30%,61.26,-9.93,6348.5,+0.00,4505.7,+51.96
6,Sparse 30% + AWQ (gs64),59.06,-13.16,2466.6,-61.15,2524.7,-14.85
7,Sparse 30% + GPTQ (gs64),60.20,-11.48,2469.5,-61.10,2186.5,-26.26
8,Sparse 40%,54.89,-19.29,6348.5,+0.00,4364.4,+47.19



=== 8B Model - Accuracy, GPU & Latency Change vs Baseline ===


,Label,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%)
0,Baseline (FP16),76.57,+0.00,15605.5,+0.00,3051.0,+0.00
1,AWQ 4-bit,73.92,-3.46,5831.3,-62.63,2599.2,-14.81
2,BNB 4-bit,75.82,-0.98,5807.4,-62.79,2691.1,-11.79
3,GPTQ 4-bit,75.51,-1.38,5836.2,-62.60,5228.4,+71.37


---
## 3. 核心比較：Sparse-only vs Quantize-only vs Sparse+Quantized

分組比較三種壓縮策略的效果。

In [42]:
df_all = pd.concat(models_chg.values(), ignore_index=True)

summary_cols = ['Model', 'Label', 'Method',
                'Accuracy (%)', 'Acc Change (%)',
                'GPU Peak (MB)', 'GPU Change (%)',
                'Time (s)', 'Latency Change (%)']

for ms in model_sizes:
    print(f'\n{"="*60}')
    print(f' Llama-{ms}-Instruct - Method Comparison')
    print(f'{"="*60}')
    subset = df_all[df_all['Model'] == ms]
    for method in ['Baseline', 'Quantize-only', 'Sparse-only', 'Sparse+Quant']:
        group = subset[subset['Method'] == method]
        if len(group) > 0:
            print(f'\n--- {method} ---')
            s = group[summary_cols].style.format(chg_fmt).hide(axis='index')
            display(style_change_gradient(s, group[summary_cols]))


 Llama-1B-Instruct - Method Comparison

--- Baseline ---


Model,Label,Method,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%)
1B,Baseline (FP16),Baseline,34.87,+0.00,2482.4,+0.00,546.3,+0.00



--- Quantize-only ---


Model,Label,Method,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%)
1B,AWQ 4-bit,Quantize-only,25.85,-25.87,1128.1,-54.55,910.4,+66.66
1B,AWQ 4-bit,Quantize-only,27.37,-21.51,1146.2,-53.82,915.4,+67.57
1B,BNB 4-bit,Quantize-only,31.69,-9.12,1145.1,-53.87,1141.8,+109.02
1B,GPTQ 4-bit,Quantize-only,30.02,-13.91,1128.6,-54.54,988.2,+80.90



--- Sparse-only ---


...

Model,Label,Method,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%)
3B,Sparse 30% + AWQ (gs64),Sparse+Quant,59.06,-13.16,2466.6,-61.15,2524.7,-14.85
3B,Sparse 30% + GPTQ (gs64),Sparse+Quant,60.20,-11.48,2469.5,-61.10,2186.5,-26.26



 Llama-8B-Instruct - Method Comparison

--- Baseline ---


Model,Label,Method,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%)
8B,Baseline (FP16),Baseline,76.57,+0.00,15605.5,+0.00,3051.0,+0.00



--- Quantize-only ---


Model,Label,Method,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%)
8B,AWQ 4-bit,Quantize-only,73.92,-3.46,5831.3,-62.63,2599.2,-14.81
8B,BNB 4-bit,Quantize-only,75.82,-0.98,5807.4,-62.79,2691.1,-11.79
8B,GPTQ 4-bit,Quantize-only,75.51,-1.38,5836.2,-62.60,5228.4,+71.37


---
## 4. Sparse 模型的量化效果比較

在已經做過 SparseGPT 的模型上，追加量化帶來的變化。以 Sparse-only 為基準。

In [43]:
def sparse_quant_table(df, sparse_label, title):
    """顯示 Sparse 模型追加量化的比較表。"""
    base_row = df[df['Label'] == sparse_label]
    if len(base_row) == 0:
        return
    base_acc = base_row['Accuracy (%)'].values[0]
    base_gpu = base_row['GPU Peak (MB)'].values[0]
    base_time = base_row['Time (s)'].values[0]

    rows = df[df['Label'].str.startswith(sparse_label)].copy()
    rows['Acc Change vs Sparse (%)'] = ((rows['Accuracy (%)'] - base_acc) / base_acc * 100).round(2)
    rows['GPU Change vs Sparse (%)'] = ((rows['GPU Peak (MB)'] - base_gpu) / base_gpu * 100).round(2)
    rows['Latency Change vs Sparse (%)'] = ((rows['Time (s)'] - base_time) / base_time * 100).round(2)

    show_cols = ['Label', 'Quantization', 'Group Size', 'Accuracy (%)',
                 'Acc Change vs Sparse (%)', 'GPU Peak (MB)',
                 'GPU Change vs Sparse (%)', 'Time (s)',
                 'Latency Change vs Sparse (%)']

    print(title)
    s = rows[show_cols].style.format({
        'Accuracy (%)': '{:.2f}', 'Acc Change vs Sparse (%)': '{:+.2f}',
        'GPU Peak (MB)': '{:.1f}', 'GPU Change vs Sparse (%)': '{:+.2f}',
        'Time (s)': '{:.1f}', 'Latency Change vs Sparse (%)': '{:+.2f}'
    }).hide(axis='index')
    display(style_change_gradient(s, rows[show_cols]))

# 自動偵測每個模型中有追加量化的 sparse level
for ms, df in models.items():
    sparse_levels = sorted(df[df['Method'] == 'Sparse-only']['Sparsity'].unique())
    for sp in sparse_levels:
        has_sq = len(df[(df['Method'] == 'Sparse+Quant') & (df['Sparsity'] == sp)]) > 0
        if has_sq:
            sparse_quant_table(df, f'Sparse {sp}', f'=== {ms}: Sparse {sp} 上追加量化 ===')
            print()

=== 1B: Sparse 20% 上追加量化 ===


Label,Quantization,Group Size,Accuracy (%),Acc Change vs Sparse (%),GPU Peak (MB),GPU Change vs Sparse (%),Time (s),Latency Change vs Sparse (%)
Sparse 20%,None,-,30.86,+0.00,2478.7,+0.00,624.0,+0.00
Sparse 20% + AWQ (gs64),AWQ,64,29.64,-3.95,1146.2,-53.76,971.4,+55.67
Sparse 20% + GPTQ (gs32),GPTQ,32,28.20,-8.62,1182.9,-52.28,1925.4,+208.55



=== 1B: Sparse 30% 上追加量化 ===


Label,Quantization,Group Size,Accuracy (%),Acc Change vs Sparse (%),GPU Peak (MB),GPU Change vs Sparse (%),Time (s),Latency Change vs Sparse (%)
Sparse 30%,None,-,26.69,+0.00,2478.7,+0.00,490.1,+0.00
Sparse 30% + AWQ (gs64),AWQ,64,23.05,-13.64,1146.2,-53.76,2319.4,+373.26
Sparse 30% + AWQ (gs128),AWQ,128,23.65,-11.39,1128.1,-54.49,1064.0,+117.10
Sparse 30% + GPTQ (gs128),GPTQ,128,23.20,-13.08,1128.6,-54.47,1025.8,+109.30
Sparse 30% + GPTQ (gs32),GPTQ,32,23.05,-13.64,1182.9,-52.28,962.1,+96.30



=== 3B: Sparse 30% 上追加量化 ===


Label,Quantization,Group Size,Accuracy (%),Acc Change vs Sparse (%),GPU Peak (MB),GPU Change vs Sparse (%),Time (s),Latency Change vs Sparse (%)
Sparse 30%,None,-,61.26,+0.00,6348.5,+0.00,4505.7,+0.00
Sparse 30% + AWQ (gs64),AWQ,64,59.06,-3.59,2466.6,-61.15,2524.7,-43.97
Sparse 30% + GPTQ (gs64),GPTQ,64,60.20,-1.73,2469.5,-61.10,2186.5,-51.47


---
## 5. 總結表格

最終整合比較表，按模型分組，按準確率降序排列。

In [44]:
final_cols = ['Model', 'Label', 'Sparsity', 'Quantization', 'Group Size',
              'Accuracy (%)', 'Acc Change (%)',
              'GPU Peak (MB)', 'GPU Change (%)',
              'Time (s)', 'Latency Change (%)',
              'Throughput (tok/s)']

final_fmt = {
    'Accuracy (%)': '{:.2f}', 'Acc Change (%)': '{:+.2f}',
    'GPU Peak (MB)': '{:.1f}', 'GPU Change (%)': '{:+.2f}',
    'Time (s)': '{:.1f}', 'Latency Change (%)': '{:+.2f}',
    'Throughput (tok/s)': '{:.1f}'
}

for ms in model_sizes:
    subset = df_all[df_all['Model'] == ms].sort_values('Accuracy (%)', ascending=False)
    print(f'\n{"="*70}')
    print(f' Llama-{ms}-Instruct - Final Summary (sorted by accuracy)')
    print(f'{"="*70}')
    s = subset[final_cols].style.format(final_fmt).hide(axis='index')
    display(style_change_gradient(s, subset[final_cols]))


 Llama-1B-Instruct - Final Summary (sorted by accuracy)


Model,Label,Sparsity,Quantization,Group Size,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%),Throughput (tok/s)
1B,Baseline (FP16),None,None,-,34.87,+0.00,2482.4,+0.00,546.3,+0.00,2738.4
1B,BNB 4-bit,None,BNB (NF4),-,31.69,-9.12,1145.1,-53.87,1141.8,+109.02,1314.8
1B,Sparse 20%,20%,None,-,30.86,-11.50,2478.7,-0.15,624.0,+14.23,2394.2
1B,GPTQ 4-bit,None,GPTQ,128,30.02,-13.91,1128.6,-54.54,988.2,+80.90,1506.5
1B,Sparse 20% + AWQ (gs64),20%,AWQ,64,29.64,-15.00,1146.2,-53.82,971.4,+77.82,1542.8
1B,Sparse 20% + GPTQ (gs32),20%,GPTQ,32,28.20,-19.13,1182.9,-52.35,1925.4,+252.46,775.4
1B,AWQ 4-bit,None,AWQ,64,27.37,-21.51,1146.2,-53.82,915.4,+67.57,1630.2
1B,Sparse 30%,30%,None,-,26.69,-23.46,2478.7,-0.15,490.1,-10.28,3031.4
1B,AWQ 4-bit,None,AWQ,128,25.85,-25.87,1128.1,-54.55,910.4,+66.66,1634.3
1B,Sparse 30% + AWQ (gs128),30%,AWQ,128,23.65,-32.18,1128.1,-54.55,1064.0,+94.78,1392.6



 Llama-3B-Instruct - Final Summary (sorted by accuracy)


Model,Label,Sparsity,Quantization,Group Size,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%),Throughput (tok/s)
3B,Baseline (FP16),None,None,-,68.01,+0.00,6348.5,+0.00,2965.1,+0.00,515.2
3B,AWQ 4-bit,None,AWQ,128,67.40,-0.90,2414.1,-61.97,2085.6,-29.66,731.9
3B,Sparse 20%,20%,None,-,65.96,-3.01,6348.5,+0.00,4570.8,+54.15,333.3
3B,BNB 4-bit,None,BNB (NF4),-,65.66,-3.46,2393.9,-62.29,2176.1,-26.61,702.1
3B,GPTQ 4-bit,None,GPTQ,128,64.97,-4.47,2417.0,-61.93,2336.5,-21.20,654.0
3B,Sparse 30%,30%,None,-,61.26,-9.93,6348.5,+0.00,4505.7,+51.96,336.4
3B,Sparse 30% + GPTQ (gs64),30%,GPTQ,64,60.20,-11.48,2469.5,-61.10,2186.5,-26.26,692.3
3B,Sparse 30% + AWQ (gs64),30%,AWQ,64,59.06,-13.16,2466.6,-61.15,2524.7,-14.85,599.4
3B,Sparse 40%,40%,None,-,54.89,-19.29,6348.5,+0.00,4364.4,+47.19,346.0



 Llama-8B-Instruct - Final Summary (sorted by accuracy)


Model,Label,Sparsity,Quantization,Group Size,Accuracy (%),Acc Change (%),GPU Peak (MB),GPU Change (%),Time (s),Latency Change (%),Throughput (tok/s)
8B,Baseline (FP16),None,None,-,76.57,+0.00,15605.5,+0.00,3051.0,+0.00,503.5
8B,BNB 4-bit,None,BNB (NF4),-,75.82,-0.98,5807.4,-62.79,2691.1,-11.79,569.7
8B,GPTQ 4-bit,None,GPTQ,128,75.51,-1.38,5836.2,-62.60,5228.4,+71.37,292.7
8B,AWQ 4-bit,None,AWQ,128,73.92,-3.46,5831.3,-62.63,2599.2,-14.81,589.4


---
## 6. Key Findings

In [45]:
print('=== Key Findings ===\n')

def pct_change(val, baseline):
    return (val - baseline) / baseline * 100

for ms, df in models.items():
    bl_acc = df.loc[df['Method'] == 'Baseline', 'Accuracy (%)'].values[0]
    print(f'【{ms} Model Observations】')
    print(f'  Baseline Accuracy: {bl_acc:.2f}%')

    qo = df[df['Method'] == 'Quantize-only']
    if len(qo) > 0:
        best = qo.loc[qo['Accuracy (%)'].idxmax()]
        print(f'  Best Quantize-only: {best["Label"]} -> {best["Accuracy (%)"]:.2f}% ({pct_change(best["Accuracy (%)"], bl_acc):+.2f}%)')

    so = df[df['Method'] == 'Sparse-only']
    for _, row in so.iterrows():
        print(f'  {row["Label"]}: {row["Accuracy (%)"]:.2f}% ({pct_change(row["Accuracy (%)"], bl_acc):+.2f}%, GPU: {row["GPU Peak (MB)"]:.0f} MB)')

    sq = df[df['Method'] == 'Sparse+Quant']
    if len(sq) > 0:
        best = sq.loc[sq['Accuracy (%)'].idxmax()]
        print(f'  Best Sparse+Quant: {best["Label"]} -> {best["Accuracy (%)"]:.2f}% ({pct_change(best["Accuracy (%)"], bl_acc):+.2f}%, GPU: {best["GPU Peak (MB)"]:.0f} MB)')
    print()

print('【Key Takeaways】')
print('  1. Sparse-only does NOT reduce GPU peak memory (same model size, just zero weights)')
print('  2. Sparse + Quantization achieves both memory reduction AND sparsity')
print('  3. Sparse 20% has notably less accuracy loss than Sparse 30%')
print('  4. Larger models are more resilient to sparsification')
print('  5. Sparse+Quant accuracy is close to Quant-only, but with additional sparsity benefits')

=== Key Findings ===

【1B Model Observations】
  Baseline Accuracy: 34.87%
  Best Quantize-only: BNB 4-bit -> 31.69% (-9.12%)
  Sparse 20%: 30.86% (-11.50%, GPU: 2479 MB)
  Sparse 30%: 26.69% (-23.46%, GPU: 2479 MB)
  Sparse 40%: 18.20% (-47.81%, GPU: 2479 MB)
  Best Sparse+Quant: Sparse 20% + AWQ (gs64) -> 29.64% (-15.00%, GPU: 1146 MB)

【3B Model Observations】
  Baseline Accuracy: 68.01%
  Best Quantize-only: AWQ 4-bit -> 67.40% (-0.90%)
  Sparse 20%: 65.96% (-3.01%, GPU: 6349 MB)
  Sparse 30%: 61.26% (-9.93%, GPU: 6349 MB)
  Sparse 40%: 54.89% (-19.29%, GPU: 6349 MB)
  Best Sparse+Quant: Sparse 30% + GPTQ (gs64) -> 60.20% (-11.48%, GPU: 2470 MB)

【8B Model Observations】
  Baseline Accuracy: 76.57%
  Best Quantize-only: BNB 4-bit -> 75.82% (-0.98%)

【Key Takeaways】
  1. Sparse-only does NOT reduce GPU peak memory (same model size, just zero weights)
  2. Sparse + Quantization achieves both memory reduction AND sparsity
  3. Sparse 20% has notably less accuracy loss than Sparse 30%
  4